# NoSQL en Databricks · 02 · Consultas y análisis
**Objetivo:** extraer tipos, expandir arrays y construir resultados sin duplicar importes.
Duración: 35 minutos. Todos los montos son USD ficticios; los cupones son metadatos y no aplican descuentos.

In [ ]:
dbutils.widgets.text("student_id", "alumno01", "01 - Identificador")
dbutils.widgets.dropdown("scale", "test", ["test", "small"], "02 - Escala NoSQL")

In [ ]:
%run ../common/config

In [ ]:
config = CourseConfig(
    spark.sql("SELECT current_catalog()").first()[0],
    dbutils.widgets.get("student_id"), dbutils.widgets.get("scale"),
)
repetitions = {"test": 1, "small": 100}[config.scale]
required = ["nosql_orders", "nosql_raw_orders", "nosql_quarantine",
            "nosql_customers", "nosql_products"]
missing = [name for name in required
           if not spark.catalog.tableExists(qualified_table(config, name))]
assert not missing, f"Faltan {missing}. Ejecutá 00_setup_datos con el mismo student_id."
assert spark.table(qualified_table(config, "nosql_orders")).count() == 12 * repetitions,             "La escala no coincide con 00_setup_datos."
spark.sql(f"USE CATALOG {quote_identifier(config.catalog)}")
spark.sql(f"USE SCHEMA {quote_identifier(config.schema)}")
print(f"Laboratorio: {config.catalog}.{config.schema}; escala {config.scale}")

## 1. Filtrar campos anidados
El operador `:` navega dentro del documento. Convertí a un tipo adecuado para comparar, ordenar o sumar.

In [ ]:
%sql
SELECT order_id, doc:shipping.country::STRING AS country,
       try_variant_get(doc, '$.total', 'decimal(12,2)') AS total,
       try_variant_get(doc, '$.risk.score', 'double') AS risk_score
FROM nosql_orders
WHERE doc:status::STRING = 'paid' AND doc:shipping.country::STRING = 'AR'
ORDER BY total DESC, order_id

## 2. Cambiar el grano: un documento → varias líneas
`variant_explode` retorna `pos`, `key`, `value`; para un array la posición empieza en cero.
La vista tendrá una fila por `(order_id, line_index)`.
Un array vacío no produce filas. Para conservar una fila también en ese caso existe `variant_explode_outer`.

In [ ]:
%sql
CREATE OR REPLACE VIEW nosql_order_items AS
WITH lines AS (
  SELECT o.order_id, e.pos AS line_index,
         try_variant_get(o.doc, '$.status', 'string') AS status,
         try_variant_get(o.doc, '$.currency', 'string') AS currency,
         try_variant_get(e.value, '$.product_id', 'string') AS product_id,
         try_variant_get(e.value, '$.category', 'string') AS category,
         try_variant_get(e.value, '$.quantity', 'bigint') AS quantity,
         try_variant_get(e.value, '$.unit_price', 'decimal(12,2)') AS unit_price
  FROM nosql_orders AS o, LATERAL variant_explode(o.doc:items) AS e
)
SELECT *, CAST(quantity * unit_price AS DECIMAL(18,2)) AS line_total
FROM lines

In [ ]:
%sql
SELECT * FROM nosql_order_items ORDER BY order_id, line_index LIMIT 20

In [ ]:
assert spark.table("nosql_order_items").count() == 18 * repetitions
assert spark.sql("""
SELECT COUNT(*) FROM (
  SELECT order_id, line_index FROM nosql_order_items
  GROUP BY order_id, line_index HAVING COUNT(*) <> 1
) AS duplicates
""").first()[0] == 0

## 3. Conservar el pedido sin líneas
Compará ambos generadores. El pedido 11 tiene un array vacío.
`variant_explode_outer` agrega una fila de valores NULL cuando no existen elementos.

In [ ]:
%sql
SELECT o.order_id, e.pos, e.value
FROM nosql_orders AS o, LATERAL variant_explode_outer(o.doc:items) AS e
WHERE o.order_id = 'O000011'

## 4. Reconciliar importes y entender el grano
El total del pedido se repite si lo unimos con cada línea. Sumar ese total después de expandir
sobrecontaría los pedidos de varias líneas. Para ventas por categoría sumá `quantity * unit_price`.
La reconciliación conserva los pedidos vacíos con un LEFT JOIN y les asigna una suma de líneas de cero.

In [ ]:
%sql
CREATE OR REPLACE TEMP VIEW nosql_reconciliation AS
WITH totals AS (
  SELECT order_id, SUM(line_total) AS item_total FROM nosql_order_items GROUP BY order_id
)
SELECT o.order_id, try_variant_get(o.doc, '$.total', 'decimal(12,2)') AS order_total,
       COALESCE(t.item_total, CAST(0 AS DECIMAL(18,2))) AS item_total
FROM nosql_orders AS o LEFT JOIN totals AS t ON o.order_id = t.order_id

In [ ]:
%sql
SELECT * FROM nosql_reconciliation ORDER BY order_id LIMIT 12

In [ ]:
assert spark.table("nosql_reconciliation").count() == 12 * repetitions
assert spark.sql("SELECT COUNT(*) FROM nosql_reconciliation WHERE order_total <> item_total").first()[0] == 0
print("Todos los totales coinciden con sus líneas, incluido el pedido vacío.")

## 5. Ventas por categoría en SQL y PySpark
Contamos solamente `paid`. Un pedido puede contribuir a más de una categoría:
sumar los conteos de pedidos por categoría no da necesariamente el número global de pedidos.

In [ ]:
%sql
SELECT category, SUM(quantity) AS units,
       CAST(SUM(line_total) AS DECIMAL(18,2)) AS amount,
       COUNT(DISTINCT order_id) AS orders
FROM nosql_order_items
WHERE status = 'paid'
GROUP BY category
ORDER BY amount DESC, category

In [ ]:
from pyspark.sql import functions as F
sales_python = (spark.table("nosql_order_items").where(F.col("status") == "paid")
                .groupBy("category")
                .agg(F.sum("quantity").alias("units"),
                     F.sum("line_total").cast("decimal(18,2)").alias("amount"),
                     F.countDistinct("order_id").alias("orders")))
display(sales_python.orderBy(F.desc("amount"), "category"))
sales_sql = spark.sql("""SELECT category, SUM(quantity) AS units,
       CAST(SUM(line_total) AS DECIMAL(18,2)) AS amount,
       COUNT(DISTINCT order_id) AS orders
FROM nosql_order_items
WHERE status = 'paid'
GROUP BY category
ORDER BY amount DESC, category
""")
assert {tuple(r) for r in sales_python.collect()} == {tuple(r) for r in sales_sql.collect()}
assert spark.sql("SELECT COUNT(*) FROM nosql_orders WHERE doc:status::STRING = 'paid'").first()[0] == 9 * repetitions

## 6. Alertas con una regla explicable
Regla: pedido pagado, total ≥ USD 1.000 y puntaje de riesgo ≥ 0,8.
Estos son candidatos para revisión; el puntaje es sintético y no prueba fraude.
Explicá por qué un pedido con puntaje alto pero total bajo queda fuera.

In [ ]:
%sql
SELECT order_id,
       try_variant_get(doc, '$.total', 'decimal(12,2)') AS total,
       try_variant_get(doc, '$.risk.score', 'double') AS risk_score
FROM nosql_orders
WHERE try_variant_get(doc, '$.status', 'string') = 'paid'
  AND try_variant_get(doc, '$.total', 'decimal(12,2)') >= 1000
  AND try_variant_get(doc, '$.risk.score', 'double') >= 0.8
ORDER BY order_id

In [ ]:
alerts = spark.sql("""SELECT order_id,
       try_variant_get(doc, '$.total', 'decimal(12,2)') AS total,
       try_variant_get(doc, '$.risk.score', 'double') AS risk_score
FROM nosql_orders
WHERE try_variant_get(doc, '$.status', 'string') = 'paid'
  AND try_variant_get(doc, '$.total', 'decimal(12,2)') >= 1000
  AND try_variant_get(doc, '$.risk.score', 'double') >= 0.8
ORDER BY order_id
""")
assert alerts.count() == 2 * repetitions
print("Consultas verificadas. Continuá con 03_desafio.")

### Para investigar
1. ¿Cuántos pedidos y cuántas líneas hay? ¿Por qué son cantidades distintas?
2. ¿Qué cambia si reemplazás `paid` por todos los estados en las ventas?
3. Calculá ventas por país de envío usando las líneas y evitando duplicar importes.
4. Mostrá un caso concreto en que sumar el total del pedido después de expandir es incorrecto.
5. ¿Qué contratos agregarías para detectar cantidades negativas, categorías ausentes o referencias rotas?